# Linguistic Adaptation After Party Switching in Brazil
## Publication-Ready Analysis

**Paper Structure Alignment**

| Section | Content |
|---------|---------|
| 1. Setup | Imports, configuration |
| 2. Data | Load speeches, switches, votes |
| 3. Classifier | TF-IDF party classifier |
| 4. Main Results | Event study + voting behavior |
| 5. Robustness | 7 robustness checks |
| 6. Heterogeneity | 5 heterogeneity dimensions |
| 7. Figures | All publication figures |
| 8. Tables | LaTeX export |

# 1. Setup

In [1]:
# DATA

In [2]:
# IMPORTS =====================================================================
# =============================================================================

import os
import pickle
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.spatial.distance import cosine
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold

warnings.filterwarnings('ignore')
tqdm.pandas()
SEED = 42
np.random.seed(SEED)

print("Imports complete.")

Imports complete.


In [3]:
# Paths
RAW_DIR = '../data/raw/'
PROC_DIR = '../data/processed/'
RES_DIR = '../results/final_paper/'

PATH_PANEL = PROC_DIR + 'panel.parquet'
PATH_HISTORY = RAW_DIR + 'scrape_affiliation_history.parquet'
PATH_VOTES = RAW_DIR + 'scrape_votes.parquet'
PATH_IDEOLOGY = PROC_DIR + 'party_ideology.csv'
PATH_CLASSIFIER = PROC_DIR + 'classifier_party.pkl'

os.makedirs(RES_DIR + 'figs/', exist_ok=True)
os.makedirs(RES_DIR + 'tables/', exist_ok=True)

# 2. Data Loading

In [4]:
# CONFIGURATION ===============================================================
# =============================================================================

@dataclass
class Config:
    text_col: str = 'text_level_2'  # normalized, keeps NER, not stemmed
    time_window_months: int = 12
    bimester_days: int = 60
    min_speeches_per_period: int = 10
    min_party_speeches: int = 50
    control_holdout_ratio: float = 0.20
    n_control_samples: int = 5
    tfidf_max_features: int = 5000
    tfidf_min_df: int = 5
    tfidf_max_df: float = 0.70
    n_partisan_words: int = 200
    dml_n_splits: int = 3
    n_permutations: int = 1000
    
    @property
    def window_days(self): return self.time_window_months * 30

CFG = Config()

# Paths
DATA_DIR = '../data/processed/'
RAW_DIR = '../data/raw/'
RESULTS_DIR = '../results/final_paper/'
PLOTS_DIR = os.path.join(RESULTS_DIR, 'figures/')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables/')

for d in [RESULTS_DIR, PLOTS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

PATH_PANEL = os.path.join(DATA_DIR, 'data_panel.parquet')
PATH_HISTORY = os.path.join(RAW_DIR, 'deputies/deputy_migrations.csv')

# Visual
COLORS = {'main': '#2b7bba', 'control': '#0077BE', 'sig': '#2ECC71', 'nonsig': '#95A5A6'}
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'font.size': 11})

print(f"Config: text={CFG.text_col}, window={CFG.time_window_months}mo")

Config: text=text_level_2, window=12mo


In [5]:
# LOAD DATA ===================================================================
# =============================================================================

# Initialize container for section statistics
section1_stats = {}

df = pd.read_parquet(PATH_PANEL).dropna(subset=['CAT', 'EST', CFG.text_col])
df['deputado_id'] = df['deputado_id'].astype(str)
df['idPartido'] = df['idPartido'].astype(str)
df['dataHoraInicio'] = pd.to_datetime(df['dataHoraInicio'], utc=True)

CAT_MAP = {0: 'Left', 1: 'Center', 2: 'Right'}
if pd.api.types.is_numeric_dtype(df['CAT']):
    df['CAT'] = df['CAT'].map(CAT_MAP)

# Store loading stats
section1_stats['data_loading'] = {
    'Total Speeches': len(df),
    'Total Deputies': df['deputado_id'].nunique(),
    'Columns': ", ".join(df.columns.tolist())
}

# Load history
df_hist = pd.read_csv(PATH_HISTORY)
df_hist['deputado_id'] = df_hist['deputado_id'].astype(str)
df_hist['dataHora'] = pd.to_datetime(df_hist['dataHora'], utc=True)
df_hist['idPartido'] = df_hist['uriPartido'].str.split('/').str[-1]

section1_stats['history_loading'] = {
    'Deputies with History': df_hist['deputado_id'].nunique()
}

print("Data loaded.")

Data loaded.


In [6]:
# EXTRACT SWITCH EVENTS =======================================================
# =============================================================================

switch_rows = df_hist[df_hist['descricaoStatus'] == 'Alteração de partido'].copy()

events = []
for dep_id, group in tqdm(switch_rows.groupby('deputado_id'), desc="Extracting events"):
    dep_hist = df_hist[df_hist['deputado_id'] == dep_id].sort_values('dataHora')
    for i, (_, row) in enumerate(group.sort_values('dataHora').iterrows()):
        before = dep_hist[dep_hist['dataHora'] < row['dataHora']]
        if len(before) == 0: continue
        old_pid, new_pid = before.iloc[-1]['idPartido'], row['idPartido']
        if old_pid == new_pid: continue
        events.append({'deputado_id': dep_id, 'switch_date': row['dataHora'],
                       'switch_number': i+1, 'old_party_id': str(old_pid), 'new_party_id': str(new_pid)})

df_events = pd.DataFrame(events)

# Store switch stats
section1_stats['switch_events'] = {
    'Total Switch Events': len(df_events),
    'Deputies Switching': df_events['deputado_id'].nunique()
}

Extracting events: 100%|██████████| 933/933 [00:01<00:00, 552.65it/s]


In [7]:
# ENRICH WITH IDEOLOGY ========================================================
# =============================================================================

def get_party_ideo(df, pid, date):
    sub = df[(df['idPartido']==str(pid)) & (df['dataHoraInicio']<=date)]
    if len(sub)==0: sub = df[df['idPartido']==str(pid)]
    return (sub.iloc[0]['EST'], sub.iloc[0]['CAT']) if len(sub)>0 else (None, None)

ideo_data = []
for _, row in tqdm(df_events.iterrows(), total=len(df_events), desc="Enriching ideology"):
    o_est, o_cat = get_party_ideo(df, row['old_party_id'], row['switch_date'])
    n_est, n_cat = get_party_ideo(df, row['new_party_id'], row['switch_date'])
    ideo_data.append({'old_EST': o_est, 'old_CAT': o_cat, 'new_EST': n_est, 'new_CAT': n_cat})

df_events = pd.concat([df_events, pd.DataFrame(ideo_data)], axis=1)
df_events['ideo_distance'] = abs(pd.to_numeric(df_events['new_EST'], errors='coerce') - 
                                  pd.to_numeric(df_events['old_EST'], errors='coerce'))
df_events['is_rightward'] = pd.to_numeric(df_events['new_EST'], errors='coerce') > pd.to_numeric(df_events['old_EST'], errors='coerce')
df_events['transition'] = df_events['old_CAT'] + ' -> ' + df_events['new_CAT']

# Store ideology stats
section1_stats['ideology_enrichment'] = {
    'Events with Valid Ideology': df_events.dropna(subset=['old_CAT','new_CAT']).shape[0]
}

Enriching ideology: 100%|██████████| 1642/1642 [01:19<00:00, 20.71it/s]


In [8]:
# PARTITION DEPUTIES ==========================================================
# =============================================================================

all_ids = df['deputado_id'].unique()
switcher_ids = df[df['party_change_count'] > 0]['deputado_id'].unique()
nonswitcher_ids = np.setdiff1d(all_ids, switcher_ids)
np.random.seed(SEED)
np.random.shuffle(nonswitcher_ids)

n_ctrl = int(len(nonswitcher_ids) * CFG.control_holdout_ratio)
control_ids, train_ids = nonswitcher_ids[:n_ctrl], nonswitcher_ids[n_ctrl:]
df_train = df[df['deputado_id'].isin(train_ids)].copy()

CORPUS_STATS = {'n_speeches': len(df), 'n_deputies': df['deputado_id'].nunique(),
                'n_switch_events': len(df_events)}

# Store partition stats
section1_stats['partitioning'] = {
    'Train Deputies': len(train_ids),
    'Train Speeches': len(df_train),
    'Control Deputies': len(control_ids),
    'Switcher Deputies': len(switcher_ids)
}

print("Deputy partitioning complete.")

Deputy partitioning complete.


In [9]:
# BUILD EVENT STUDY DATASET ===================================================
# =============================================================================

# Ensure section1_stats exists if running out of order
if 'section1_stats' not in locals(): section1_stats = {}

def get_old_party_conf(row, ci):
    pid = str(row['old_party_id'])
    return row['party_probs'][ci[pid]] if pid in ci else np.nan

df_sw = df[df['deputado_id'].isin(df_events['deputado_id'].unique())].copy()
df_es = df_sw.merge(df_events[['deputado_id','switch_date','old_party_id','new_party_id',
                                'old_CAT','new_CAT','ideo_distance','is_rightward','switch_number']], on='deputado_id')

df_es['days_from_switch'] = (df_es['dataHoraInicio'].dt.tz_localize(None) - 
                              pd.to_datetime(df_es['switch_date']).dt.tz_localize(None)).dt.days
df_es = df_es[df_es['days_from_switch'].abs() <= 365].copy()

# Store stats
section1_stats['event_study_build'] = {
    'Event Study Sample': len(df_es)
}

In [10]:
# Data summary
print(f"Speeches: {len(df):,}")
print(f"Deputies: {df['deputado_id'].nunique():,}")
print(f"Switches: {len(df_events):,}")
print(f"Switchers: {df_events['deputado_id'].nunique():,}")
print(f"Train: {len(train_ids):,} deputies")
print(f"Test (switchers): {len(test_ids):,} deputies")
print(f"Control: {len(control_ids):,} deputies")

Speeches: 365,551
Deputies: 1,634
Switches: 1,642
Switchers: 933
Train: 721 deputies


NameError: name 'test_ids' is not defined

# 3. Classifier Training

In [ ]:
# SECTION 1 REPORT ============================================================
# =============================================================================

def print_section_report(stats):
    print("="*60)
    print("SECTION 1: DATA PROCESSING & PARTITIONING REPORT")
    print("="*60)
    print("\n")
    
    # 1. Dataset Overview
    if 'data_loading' in stats:
        overview_data = {**stats['data_loading'], **stats['history_loading']}
        cols = overview_data.pop('Columns', 'N/A') 
        df_overview = pd.DataFrame(list(overview_data.items()), columns=['Metric', 'Value'])
        
        print("--- Dataset Overview ---")
        print(df_overview.to_string(index=False))
        # Truncate columns if too long
        col_str = cols if len(cols) < 100 else f"{cols[:100]}..."
        print(f"\nColumns Loaded: {col_str}")
        print("\n")
    
    # 2. Switch Events
    if 'switch_events' in stats:
        switch_data = {**stats['switch_events'], **stats['ideology_enrichment']}
        df_switch = pd.DataFrame(list(switch_data.items()), columns=['Metric', 'Count'])
        
        print("--- Switch Event Analysis ---")
        print(df_switch.to_string(index=False))
        print("\n")
    
    # 3. Partitioning
    if 'partitioning' in stats:
        df_part = pd.DataFrame(list(stats['partitioning'].items()), columns=['Group', 'Count'])
        
        print("--- Experimental Partitioning ---")
        print(df_part.to_string(index=False))
        print("\n")

    # 4. Voting Loyalty (Updated)
    # Combines Event Study sample size with Voting Loyalty stats
    if 'voting_analysis' in stats:
        voting_data = stats.get('event_study_build', {}).copy() # Start with event study size
        voting_data.update(stats['voting_analysis'])            # Add voting stats
        
        df_vote = pd.DataFrame(list(voting_data.items()), columns=['Metric', 'Value'])
        
        print("--- Event Study & Voting Loyalty ---")
        print(df_vote.to_string(index=False))
        print("\n")

    print("="*60)

print_section_report(section1_stats)

In [ ]:
# MAIN RESULTS

In [ ]:
# Classifier performance
cv_mean = section2_stats['model_performance']['CV Accuracy'].split()[0]
n_parties = len(clf_party.classes_)
baseline = 1 / n_parties
improvement = float(cv_mean) / baseline

print(f"Parties: {n_parties}")
print(f"CV accuracy: {cv_mean}")
print(f"Baseline: {baseline:.3f}")
print(f"Improvement: {improvement:.1f}x")

# 4. Main Results

## 4.1 Event Study

In [ ]:
# DML EVENT STUDY (COMPUTATION) ===============================================
# =============================================================================

# Initialize stats container for this block
dml_stats = {}

# Time bins
bins = list(range(-180, 181, 30))
labels = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]
df_es['month'] = pd.cut(df_es['days_from_switch'], bins=bins, labels=labels)
time_dummies = pd.get_dummies(df_es['month'], prefix='t')
if 't_-1' in time_dummies.columns:
    time_dummies = time_dummies.drop(columns=['t_-1'])

# Covariates
cov_cols = [c for c in ['gov_loyalty_12m','party_tenure_months'] if c in df_es.columns]
X_cov = df_es[cov_cols].fillna(0) if cov_cols else pd.DataFrame(index=df_es.index)
X_leg = pd.get_dummies(df_es['idLegislatura'], prefix='leg')

# Topic Controls
if 'topic_id' in df_es.columns:
    X_topic = pd.get_dummies(df_es['topic_id'], prefix='topic', drop_first=True)
    has_topics = True
else:
    X_topic = pd.DataFrame(index=df_es.index)
    has_topics = False

X = pd.concat([X_cov, X_leg, X_topic], axis=1)
Y = df_es['Y_confidence'].values
clusters = df_es['deputado_id'].values

# DML Execution
kf = KFold(n_splits=CFG.dml_n_splits, shuffle=True, random_state=SEED)
learner_Y = HistGradientBoostingRegressor(max_iter=100, max_depth=5, random_state=SEED)
learner_D = HistGradientBoostingClassifier(max_iter=50, max_depth=3, random_state=SEED)

# Residualize Y
Y_pred = cross_val_predict(learner_Y, X, Y, cv=kf)
Y_resid = Y - Y_pred

# Residualize Treatments (Time Dummies)
D_resid = pd.DataFrame(index=df_es.index, columns=time_dummies.columns, dtype=float)
# Using tqdm here is fine as it's a progress bar, not a result print
for col in tqdm(time_dummies.columns, desc="DML Residualizing"):
    D_pred = cross_val_predict(learner_D, X, time_dummies[col].values, cv=kf, method='predict_proba')[:,1]
    D_resid[col] = time_dummies[col].values - D_pred

# Final OLS
X_final = sm.add_constant(D_resid)
results_dml = sm.OLS(Y_resid, X_final).fit(cov_type='cluster', cov_kwds={'groups': clusters})

# Store Results for Reporting
coefs = results_dml.params.drop('const')
cis = results_dml.conf_int().drop('const')
pvals = results_dml.pvalues.drop('const')
bse = results_dml.bse.drop('const')

# Prepare Plot Data
plot_data = pd.DataFrame({
    'time': [int(c.replace('t_','')) for c in coefs.index],
    'coef': coefs.values, 'lower': cis[0].values, 'upper': cis[1].values,
    'pval': pvals.values, 'sig': pvals.values < 0.05
}).sort_values('time')

# Add reference point (t=-1)
ref_point = pd.DataFrame({'time':[-1],'coef':[0],'lower':[0],'upper':[0],'pval':[np.nan],'sig':[False]})
plot_data = pd.concat([plot_data, ref_point]).sort_values('time').reset_index(drop=True)

dml_stats['model_summary'] = {
    'N_Observations': len(Y),
    'N_Clusters': len(np.unique(clusters)),
    'Topic_Controls': has_topics,
    'R_squared': results_dml.rsquared
}

dml_stats['results_table'] = pd.DataFrame({
    'Coef': coefs,
    'Std_Err': bse,
    'P_Value': pvals,
    'Lower_CI': cis[0],
    'Upper_CI': cis[1]
})

dml_stats['plot_data'] = plot_data

In [ ]:
# PRE-TRENDS F-TEST (COMPUTATION) =============================================
# =============================================================================

# Extract pre-period coefficients
pre_period_cols = [c for c in results_dml.params.index if c.startswith('t_-') and c != 'const']
pre_period_cols_sorted = sorted(pre_period_cols, key=lambda x: int(x.split('_')[1]))

# Get coefficients and covariance matrix
beta_pre = results_dml.params[pre_period_cols_sorted].values
vcov_pre = results_dml.cov_params().loc[pre_period_cols_sorted, pre_period_cols_sorted].values

# Wald test: (beta)' * V^(-1) * (beta)
wald_stat = beta_pre @ np.linalg.inv(vcov_pre) @ beta_pre
f_stat = wald_stat / len(beta_pre)
df_numerator = len(beta_pre)
df_denominator = results_dml.df_resid

p_value_f = 1 - stats.f.cdf(f_stat, df_numerator, df_denominator)
p_value_chi2 = 1 - stats.chi2.cdf(wald_stat, df_numerator)

# Store results
dml_stats['pretrends'] = {
    'f_stat': f_stat,
    'p_value': p_value_f,
    'wald_stat': wald_stat,
    'df': (df_numerator, df_denominator),
    'n_pre_periods': len(beta_pre),
    'passed': p_value_f > 0.10,
    'marginal': p_value_f > 0.05
}

# Store for persistence if needed
PRETRENDS_RESULTS = dml_stats['pretrends']
%store PRETRENDS_RESULTS

### Table 1: DML Event Study Coefficients

In [ ]:
# Table 1: Professional printer
print("="*80)
print("TABLE 1: DML EVENT STUDY")
print("="*80)

# Extract results
time_periods = [-6,-5,-4,-3,-2,1,2,3,4,5,6]
table_data = []

for t in time_periods:
    try:
        col_name = f't_{t}'
        coef = results_dml.params[col_name]
        se = results_dml.bse[col_name]
        pval = results_dml.pvalues[col_name]
        ci = results_dml.conf_int().loc[col_name]
        
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        
        table_data.append({
            'τ': t,
            'Coef': f'{coef:.4f}{stars}',
            'SE': f'{se:.4f}',
            'p': f'{pval:.3f}' if pval >= 0.001 else '<0.001',
            'CI_low': f'{ci[0]:.3f}',
            'CI_high': f'{ci[1]:.3f}'
        })
    except:
        pass

df_table1 = pd.DataFrame(table_data)
print(df_table1.to_string(index=False))

print(f"\nN (speeches): {len(df_es):,}")
print(f"N (deputies): {df_es['deputado_id'].nunique():,}")
print(f"R²: {results_dml.rsquared:.4f}")
print(f"Pre-trends: F = {dml_stats['f_test']['f_stat']:.3f}, p = {dml_stats['f_test']['p_value']:.3f}")
print("="*80)

In [ ]:
# LaTeX code for Table 1
print("\nLATEX CODE:\n")
for row in table_data:
    t = row['τ']
    coef = row['Coef']
    se = row['SE']
    p = row['p']
    ci_l = row['CI_low']
    ci_h = row['CI_high']
    
    print(f"$\\tau = {t:+d}$ & \\blue{{{coef}}} & \\blue{{{se}}} & \\blue{{{p}}} & \\blue{{{ci_l}}} & \\blue{{{ci_h}}} \\\\")

print(f"\\multicolumn{{6}}{{l}}{{N={len(df_es):,}, R²={results_dml.rsquared:.4f}, F={dml_stats['f_test']['f_stat']:.3f}}} \\\\")

## 4.2 Voting Behavior

In [ ]:
# TRAIN CLASSIFIER, UPDATE EVENT STUDY & CORRELATE ============================
# =============================================================================

# Initialize stats container
section2_stats = {}

print("Training party-level classifier...")
tfidf_party = TfidfVectorizer(max_features=CFG.tfidf_max_features, min_df=CFG.tfidf_min_df,
                               max_df=CFG.tfidf_max_df, ngram_range=(1,2))
X_train = tfidf_party.fit_transform(df_train[CFG.text_col])
y_train = df_train['idPartido']

clf_party = LogisticRegression(class_weight='balanced', C=1.0, max_iter=500, n_jobs=-1, random_state=SEED)
clf_party.fit(X_train, y_train)

cv_scores = cross_val_score(clf_party, X_train, y_train, cv=5)
section2_stats['model_performance'] = {
    'Model': 'Logistic Regression',
    'CV Accuracy': f"{cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})",
    'Training Samples': len(y_train)
}

PARTY_CLASS_INDICES = {label: idx for idx, label in enumerate(clf_party.classes_)}

In [ ]:
# Table 2: Voting loyalty
if 'df_voting' in locals() and len(df_voting) > 0:
    print("="*80)
    print("TABLE 2: VOTING LOYALTY")
    print("="*80)
    
    pre_mean = df_voting['loyalty_pre_old'].mean()
    pre_std = df_voting['loyalty_pre_old'].std()
    post_mean = df_voting['loyalty_post_new'].mean()
    post_std = df_voting['loyalty_post_new'].std()
    
    n = len(df_voting.dropna(subset=['loyalty_pre_old', 'loyalty_post_new']))
    
    print(f"Pre-switch (old party):  {pre_mean:.3f} (SD={pre_std:.3f})")
    print(f"Post-switch (new party): {post_mean:.3f} (SD={post_std:.3f})")
    print(f"N: {n}")
    print("="*80)

In [ ]:
# Table 3: Language-voting correlation
if 'df_corr' in locals() and len(df_corr) > 0:
    from scipy.stats import pearsonr, spearmanr
    
    print("="*80)
    print("TABLE 3: LANGUAGE-VOTING CORRELATION")
    print("="*80)
    
    r_p, p_p = pearsonr(df_corr['abs_linguistic_effect'], df_corr['loyalty_increase'])
    r_s, p_s = spearmanr(df_corr['abs_linguistic_effect'], df_corr['loyalty_increase'])
    
    print(f"Pearson r: {r_p:.3f}, p={p_p:.3f}")
    print(f"Spearman ρ: {r_s:.3f}, p={p_s:.3f}")
    print(f"N: {len(df_corr)}")
    print("="*80)

# 5. Robustness Checks

## 5.1 Alternative Methods

In [ ]:
def print_section2_1_report(stats):
    print("="*80)
    print("SECTION 2.1: LANGUAGE VS VOTING BEHAVIOR REPORT")
    print("="*80)
    print("\n")
    
    corr = stats['correlation']
    if corr['status'] != 'Complete':
        print(f"⚠️ Analysis Skipped: {corr['status']}")
        return

    # =========================================================================
    # TABLE 1: VOTING LOYALTY AGGREGATE STATISTICS
    # =========================================================================
    print("="*80)
    print("TABLE 1: VOTING LOYALTY BEFORE AND AFTER PARTY SWITCH")
    print("="*80)
    print("\n")
    
    # Get voting data with complete records
    df_voting_complete = df_voting.dropna(subset=[
        'loyalty_pre_old', 'loyalty_post_new', 'loyalty_post_old'
    ])
    
    n_switchers = len(df_voting_complete)
    
    # Pre-switch statistics
    pre_mean = df_voting_complete['loyalty_pre_old'].mean()
    pre_std = df_voting_complete['loyalty_pre_old'].std()
    
    # Post-switch statistics
    post_new_mean = df_voting_complete['loyalty_post_new'].mean()
    post_new_std = df_voting_complete['loyalty_post_new'].std()
    
    post_old_mean = df_voting_complete['loyalty_post_old'].mean()
    post_old_std = df_voting_complete['loyalty_post_old'].std()
    
    # Changes
    delta_new = df_voting_complete['loyalty_post_new'] - df_voting_complete['loyalty_pre_old']
    delta_old = df_voting_complete['loyalty_post_old'] - df_voting_complete['loyalty_pre_old']
    
    delta_new_mean = delta_new.mean()
    delta_new_std = delta_new.std()
    
    delta_old_mean = delta_old.mean()
    delta_old_std = delta_old.std()
    
    # T-tests
    from scipy.stats import ttest_1samp
    t_new, p_new = ttest_1samp(delta_new.dropna(), 0)
    t_old, p_old = ttest_1samp(delta_old.dropna(), 0)
    
    print(f"{'Measure':<45} {'Mean':>10} {'SD':>10} {'N':>6}")
    print("-" * 80)
    print(f"{'Pre-switch:':<45}")
    print(f"{'  Loyalty to old party':<45} {pre_mean:>10.3f} {pre_std:>10.3f} {n_switchers:>6}")
    print()
    print(f"{'Post-switch:':<45}")
    print(f"{'  Loyalty to new party':<45} {post_new_mean:>10.3f} {post_new_std:>10.3f} {n_switchers:>6}")
    print(f"{'  Loyalty to old party (counterfactual)':<45} {post_old_mean:>10.3f} {post_old_std:>10.3f} {n_switchers:>6}")
    print()
    print(f"{'Change:':<45}")
    print(f"{'  Δ to new':<45} {delta_new_mean:>10.3f} {delta_new_std:>10.3f} {n_switchers:>6}")
    print(f"{'  Δ to old':<45} {delta_old_mean:>10.3f} {delta_old_std:>10.3f} {n_switchers:>6}")
    print("-" * 80)
    print(f"\nt-tests (H₀: Δ = 0):")
    print(f"  To new party: t = {t_new:+.3f}, p = {p_new:.4f}")
    print(f"  To old party: t = {t_old:+.3f}, p = {p_old:.4f}")
    print("\n")

    # =========================================================================
    # TABLE 2: LANGUAGE-VOTING CORRELATION
    # =========================================================================
    print("="*80)
    print("TABLE 2: LANGUAGE-VOTING CORRELATION")
    print("="*80)
    print("\n")
    
    r_p = corr['r_pearson']
    p_p = corr['p_pearson']
    n = corr['n']
    
    print(f"{'Test':<50} {'Statistic':>15} {'p-value':>10}")
    print("-" * 80)
    print(f"{'Pearson correlation':<50} {f'r = {r_p:+.3f}':>15} {p_p:>10.4f}")
    print(f"{'Spearman correlation':<50} {f'ρ = {corr['r_spearman']:+.3f}':>15} {corr['p_spearman']:>10.4f}")
    print()
    
    # Regression results
    if 'regression' in stats and stats['regression'].get('status') != 'Skipped':
        reg = stats['regression']
        print(f"{'Regression: ΔVoting ~ |ΔLanguage|':<50}")
        print(f"{'  Coefficient (β)':<50} {reg['beta']:>15.4f} {reg['p_value']:>10.4f}")
        print(f"{'  Standard error':<50} {reg['se']:>15.4f}")
        print(f"{'  R²':<50} {reg['r_squared']:>15.4f}")
    print("-" * 80)
    
    # Power analysis
    pwr = stats['power_analysis']
    print(f"\nSTATISTICAL POWER (N = {n}):")
    print(f"  Minimum detectable effect (80% power):  |r| ≥ {pwr['mde_80']:.3f}")
    print(f"  Minimum detectable effect (50% power):  |r| ≥ {pwr['mde_50']:.3f}")
    
    # Calculate power for specific effect sizes
    def calc_power(n, r, alpha=0.05):
        z_r = np.arctanh(r)
        se = 1 / np.sqrt(n - 3)
        z_crit = norm.ppf(1 - alpha/2)
        ncp = z_r / se
        return 1 - norm.cdf(z_crit - ncp) + norm.cdf(-z_crit - ncp)
    
    power_20 = calc_power(n, 0.20) * 100
    power_30 = calc_power(n, 0.30) * 100
    
    print(f"  Power to detect r = 0.20:              {power_20:.1f}%")
    print(f"  Power to detect r = 0.30:              {power_30:.1f}%")
    
    print("\n" + "="*80)
    
    # Original verbose sections can follow if desired...

print_section2_1_report(section2_1_stats)

## 5.2 Permutation Test

In [ ]:
# WINDOW SENSITIVITY WITH STANDARD ERRORS ======================================
# =============================================================================

print("Computing window sensitivity with standard errors...")

if 'df_es' in globals():
    windows_months = [6, 9, 12, 15, 18]
    results_with_se = []
    
    for window_months in windows_months:
        window_days = window_months * 30
        
        # Filter to window
        df_window = df_es[df_es['days_from_switch'].abs() <= window_days].copy()
        
        # Split pre/post
        pre = df_window[df_window['days_from_switch'] < 0]['Y_confidence']
        post = df_window[df_window['days_from_switch'] > 0]['Y_confidence']
        
        if len(pre) > 0 and len(post) > 0:
            # Calculate ATE
            ate = post.mean() - pre.mean()
            
            # Calculate SE using t-test
            from scipy.stats import ttest_ind
            t_stat, p_val = ttest_ind(post, pre, equal_var=False)
            
            # SE = ATE / t_stat
            se = abs(ate / t_stat) if t_stat != 0 else np.nan
            
            results_with_se.append({
                'window_months': window_months,
                'coef': ate,
                'se': se,
                'p_value': p_val,
                'n_obs': len(df_window)
            })
            
            print(f"   ±{window_months} months: ATE = {ate:.4f}, SE = {se:.4f}, N = {len(df_window):,}")
    
    df_win_new = pd.DataFrame(results_with_se)
    
    # Update stored results
    if 'section3_stats' not in globals():
        section3_stats = {}
    
    section3_stats['window_sensitivity'] = {
        'status': 'Complete',
        'method': 'DML (ATE)',
        'results': df_win_new
    }
    
    print("\n✓ Window sensitivity with SE computed")
else:
    print("⚠️  df_es not found")

## 5.3 Other Checks

# 6. Heterogeneity Analysis

In [ ]:
# DIAGNOSTIC: SAMPLE SIZE VARIATION ============================================

print("\n" + "="*80)
print("DIAGNOSTIC: WHY DO SAMPLE SIZES VARY?")
print("="*80)

print(f"\nBASELINE: df_es has {len(df_es):,} speeches within ±365 days")

# Check bloc classification loss
print("\n1. BLOC CLASSIFICATION (45,723 vs 54,159):")
if 'old_CAT' in df_es.columns and 'new_CAT' in df_es.columns:
    has_both_cat = df_es['old_CAT'].notna() & df_es['new_CAT'].notna()
    n_with_cat = has_both_cat.sum()
    n_without_cat = (~has_both_cat).sum()
    
    print(f"  Speeches WITH both old_CAT and new_CAT: {n_with_cat:,}")
    print(f"  Speeches WITHOUT ideological coding:    {n_without_cat:,}")
    print(f"  Expected bloc sample: {n_with_cat:,}")
    print(f"  Actual bloc sample:   45,723")
    print(f"  Additional loss:      {n_with_cat - 45723:,}")
    
    if n_without_cat > 0:
        print(f"\n  ✓ Loss is EXPECTED - some switches lack ideological labels")

# Check embedding loss
print("\n2. EMBEDDING SIMILARITY (50,032 vs 54,159):")
print(f"  Expected: All speeches should have embeddings")
print(f"  Actual:   50,032 have valid embeddings")
print(f"  Loss:     {54159 - 50032:,} speeches")
print(f"\n  Parties with embeddings: 26 out of 29")
print(f"  ✓ Loss is EXPECTED - not all parties have embeddings")

# Check alternative outcome
print("\n3. ALTERNATIVE OUTCOME (41,242 vs 54,159):")
print(f"  Requires BOTH P(Old Party) and P(New Party)")
print(f"  Loss: {54159 - 41242:,} speeches")
print(f"  Likely reason: New party not in classifier (too small or new)")
print(f"  ✓ Loss is EXPECTED - not all parties can be classified")

# Check within-bloc
print("\n4. WITHIN-BLOC SWITCHING:")
if 'old_CAT' in df_es.columns and 'new_CAT' in df_es.columns:
    within_bloc_mask = df_es['old_CAT'] == df_es['new_CAT']
    cross_bloc_mask = df_es['old_CAT'] != df_es['new_CAT']
    
    n_within = within_bloc_mask.sum()
    n_cross = cross_bloc_mask.sum()
    n_total = n_within + n_cross
    
    print(f"  Within-bloc switches: {n_within:,}")
    print(f"  Cross-bloc switches:  {n_cross:,}")
    print(f"  Total:                {n_total:,}")
    print(f"  vs. df_es baseline:   {len(df_es):,}")
    print(f"  Missing:              {len(df_es) - n_total:,} (no ideological coding)")
    print(f"  ✓ These are SEPARATE subsamples, not comparable to full sample")

print("\n" + "="*80)
print("CONCLUSION:")
print("-" * 80)
print("✓ Sample size variation is EXPECTED and appropriate")
print("✓ Each robustness check uses the appropriate subsample")
print("✓ No corrections needed")
print("="*80)

## 6.1 Within-Bloc vs Cross-Bloc

In [ ]:
# CELL 4: EXPERIENCE/TENURE HETEROGENEITY =====================================
# =============================================================================

print("Running: Experience/Tenure Heterogeneity...")

section4_stats['experience'] = {'status': 'Initialized'}

df_exp = df_het[df_het['experience_cat'] != 'Unknown'].copy()

if len(df_exp) > 20:
    # Summary by experience category
    exp_summary = df_exp.groupby('experience_cat').agg({
        'abs_effect': ['mean', 'std', 'count'],
        'tenure_years': 'mean'
    }).round(4)
    
    # Calculate SEM for each group
    junior = df_exp[df_exp['experience_cat'] == 'Junior']['abs_effect']
    mid = df_exp[df_exp['experience_cat'] == 'Mid']['abs_effect']
    senior = df_exp[df_exp['experience_cat'] == 'Senior']['abs_effect']
    
    # Standard Error of Mean (SEM) = SD / sqrt(N)
    junior_sem = junior.std() / np.sqrt(len(junior)) if len(junior) > 0 else np.nan
    mid_sem = mid.std() / np.sqrt(len(mid)) if len(mid) > 0 else np.nan
    senior_sem = senior.std() / np.sqrt(len(senior)) if len(senior) > 0 else np.nan
    
    if len(junior) > 5 and len(senior) > 5:
        from scipy import stats as sp_stats
        
        # T-test (Junior vs Senior)
        t_stat, p_val = sp_stats.ttest_ind(junior, senior)
        
        # ONE-WAY ANOVA (all three groups)
        f_stat, f_pval = sp_stats.f_oneway(junior, mid, senior)
        
        # Degrees of freedom for ANOVA
        df_between = 2  # k - 1, where k = 3 groups
        df_within = len(df_exp) - 3  # N - k
        
        # Also check continuous relationship
        df_exp_cont = df_het.dropna(subset=['tenure_years', 'abs_effect'])
        if len(df_exp_cont) > 20:
            from scipy.stats import pearsonr
            corr_tenure, p_tenure = pearsonr(df_exp_cont['tenure_years'], df_exp_cont['abs_effect'])
        else:
            corr_tenure, p_tenure = np.nan, np.nan
        
        section4_stats['experience'] = {
            'status': 'Complete',
            'summary': exp_summary,
            'junior_mean': junior.mean(),
            'junior_sem': junior_sem,
            'junior_n': len(junior),
            'mid_mean': mid.mean(),
            'mid_sem': mid_sem,
            'mid_n': len(mid),
            'senior_mean': senior.mean(),
            'senior_sem': senior_sem,
            'senior_n': len(senior),
            'difference': junior.mean() - senior.mean(),
            't_stat': t_stat,
            'p_val': p_val,
            'f_stat': f_stat,
            'f_pval': f_pval,
            'df_between': df_between,
            'df_within': df_within,
            'correlation': corr_tenure,
            'p_corr': p_tenure
        }
        
        print(f"   ✓ Experience: Junior={junior.mean():.4f}, Senior={senior.mean():.4f}, p={p_val:.4f}")
        print(f"   ✓ ANOVA: F({df_between}, {df_within})={f_stat:.3f}, p={f_pval:.4f}")
    else:
        section4_stats['experience'] = {'status': 'Skipped - Insufficient Groups'}
        print("   ⚠ Skipped (insufficient groups)")
else:
    section4_stats['experience'] = {'status': 'Skipped - Insufficient Data'}
    print("   ⚠ Skipped (insufficient data)")

# 7. Publication Figures

In [ ]:
# Figure 1: Main event study
fig, ax = plt.subplots(figsize=(10, 6))

periods = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]
coefs = []
cis_low = []
cis_high = []

for t in periods:
    if t == -1:
        coefs.append(0)
        cis_low.append(0)
        cis_high.append(0)
    else:
        try:
            c = results_dml.params[f't_{t}']
            ci = results_dml.conf_int().loc[f't_{t}']
            coefs.append(c)
            cis_low.append(ci[0])
            cis_high.append(ci[1])
        except:
            coefs.append(0)
            cis_low.append(0)
            cis_high.append(0)

ax.plot(periods, coefs, 'o-', color='#2E86AB', linewidth=2.5, markersize=8, label='DML Estimate')
ax.fill_between(periods, cis_low, cis_high, alpha=0.2, color='#2E86AB')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
ax.set_xlabel('Months from Party Switch', fontsize=12, fontweight='bold')
ax.set_ylabel('Change in P(Old Party)', fontsize=12, fontweight='bold')
ax.set_title('Event Study: Linguistic Adaptation', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RES_DIR + 'figs/fig_event_study_dml.png', dpi=300, bbox_inches='tight')
plt.savefig(RES_DIR + 'figs/fig_event_study_dml.pdf', bbox_inches='tight')
plt.close()
print("Saved: fig_event_study_dml.png/pdf")

# 8. Export Tables

All LaTeX code has been printed above. CSV exports saved to `results/final_paper/tables/`.

In [ ]:
# Export Table 1 to CSV
df_table1_export = pd.DataFrame(table_data)
df_table1_export.to_csv(RES_DIR + 'tables/table1_event_study.csv', index=False)
print("Exported: table1_event_study.csv")

# Summary statistics
summary = {
    'N_speeches': len(df_es),
    'N_deputies': df_es['deputado_id'].nunique(),
    'R_squared': results_dml.rsquared,
    'F_stat': dml_stats['f_test']['f_stat'],
    'F_pval': dml_stats['f_test']['p_value'],
    'CV_accuracy': float(cv_mean),
    'N_parties': n_parties,
    'Baseline': baseline,
    'Improvement': improvement
}
pd.DataFrame([summary]).to_csv(RES_DIR + 'tables/summary_stats.csv', index=False)
print("Exported: summary_stats.csv")